# scRNA Demultiplexing
Vireo-based cell-line demultiplexing results, donor assignment quality, concordance profiles,
and reference-guided validation using Pool_ctr.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import os, glob, sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH   = 'results/variants.db'
DEMUX_DIR = 'results/demux'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

def load_demux(run_id):
    fa = pd.read_csv(f'{DEMUX_DIR}/{run_id}/final_assignments.tsv', sep='\t')
    dm_path = f'{DEMUX_DIR}/{run_id}/donor_matches.tsv'
    dm = pd.read_csv(dm_path, sep='\t') if os.path.exists(dm_path) else pd.DataFrame()
    return fa, dm

demux_runs = sorted(
    os.path.basename(os.path.dirname(p))
    for p in glob.glob(f'{DEMUX_DIR}/*/final_assignments.tsv')
)
print(f'Pools found: {demux_runs}')
if not demux_runs:
    print('No demux runs found — run the scrna pipeline first.')

## scRNA demultiplexing results
Cell composition, doublet rate, Vireo donor probabilities, and concordance with DB runs.

In [ ]:
if not demux_runs:
    print('No demux runs available.')
else:
    demux_selector = widgets.Dropdown(
        options=demux_runs,
        description='Demux run:',
        layout=widgets.Layout(width='350px')
    )

    SINGLET_SOURCES = {'db_match', 'guided_match'}

    def render_scrna(run_id):
        fa, dm = load_demux(run_id)

        # ── 1. Cell composition bar ──────────────────────────────────────────
        singlets   = fa[fa['source'].isin(SINGLET_SOURCES)].copy()
        vireo_only = fa[fa['source'] == 'vireo_only'].copy()

        db_order = (
            singlets.groupby('cell_line').size()
            .sort_values(ascending=False).index.tolist()
        )
        vireo_order = (
            vireo_only.groupby('cell_line').size()
            .sort_values(ascending=False).index.tolist()
        )
        line_order = db_order + vireo_order

        fig1 = go.Figure()
        colors_conf = {'high': '#2d6a4f', 'low': '#f4a261', 'guided': '#2d6a4f'}
        for conf, color in colors_conf.items():
            sub = singlets[singlets['match_confidence'] == conf]
            if sub.empty:
                continue
            counts = sub['cell_line'].value_counts().reindex(db_order, fill_value=0)
            label = 'Guided (Vireo)' if conf == 'guided' else f'DB match ({conf})'
            fig1.add_trace(go.Bar(x=counts.index, y=counts.values,
                                  name=label, marker_color=color))

        if len(vireo_only):
            vo_counts = vireo_only['cell_line'].value_counts().reindex(vireo_order, fill_value=0)
            fig1.add_trace(go.Bar(x=vo_counts.index, y=vo_counts.values,
                                  name='vireo only (unmatched)', marker_color='#6d6875'))

        for status, color in [('doublet', '#e63946'), ('unassigned', '#adb5bd')]:
            fig1.add_trace(go.Bar(x=[status], y=[(fa['source'] == status).sum()],
                                  name=status, marker_color=color))

        fig1.update_layout(
            barmode='stack',
            title=f'Cell composition — {run_id}  ({len(fa):,} cells)',
            xaxis_title='Cell line / donor', yaxis_title='Cell count',
            height=420, legend_title='Assignment'
        )
        display(fig1)

        # ── 2. Doublet pair heatmap ───────────────────────────────────────────
        doublets = fa[fa['source'] == 'doublet'].copy()
        dbl_mat  = pd.DataFrame(0, index=line_order, columns=line_order)

        for pair_str in doublets['cell_line'].str.replace('^doublet:', '', regex=True):
            parts = [p.strip() for p in pair_str.split('+')]
            if len(parts) == 2:
                a, b = parts
                if a in dbl_mat.index and b in dbl_mat.columns:
                    dbl_mat.loc[a, b] += 1
                    dbl_mat.loc[b, a] += 1

        dbl_vals = dbl_mat.values.astype(float)
        dbl_text = dbl_mat.values.astype(str)
        tril = np.tril_indices(len(dbl_vals), k=-1)
        dbl_vals[tril] = np.nan
        dbl_text[tril] = ''

        fig2 = go.Figure(go.Heatmap(
            z=dbl_vals, x=dbl_mat.columns.tolist(), y=dbl_mat.index.tolist(),
            colorscale='Reds', text=dbl_text, texttemplate='%{text}',
            hovertemplate='%{y} + %{x}<br>Doublets: %{z}<extra></extra>'
        ))
        fig2.update_layout(
            title=f'Doublet pair counts — {run_id}  ({len(doublets):,} doublets)',
            height=max(420, 22 * len(line_order)), xaxis_tickangle=45
        )
        display(fig2)

        # ── 3. Vireo probability distribution ────────────────────────────────
        prob_df = pd.concat([singlets, vireo_only]).copy()
        prob_df['vireo_prob_max'] = pd.to_numeric(prob_df['vireo_prob_max'], errors='coerce')

        fig3 = go.Figure()
        for line in line_order:
            sub = prob_df[prob_df['cell_line'] == line]['vireo_prob_max'].dropna()
            if len(sub) == 0:
                continue
            fig3.add_trace(go.Violin(
                y=sub, name=line, box_visible=True,
                meanline_visible=True, points=False,
                line_color='#6d6875' if line.startswith('vireo:') else None
            ))
        fig3.update_layout(
            title=f'Vireo assignment probability per cell line — {run_id}',
            yaxis_title='prob_max (Vireo confidence)',
            height=420, showlegend=False
        )
        display(fig3)

        # ── 4. Vireo → DB concordance & gap bars ─────────────────────────────
        _MIN_CONCORDANCE = 0.80
        _MIN_GAP = 0.10
        if not dm.empty and 'concordance' in dm.columns:
            dm['concordance'] = pd.to_numeric(dm['concordance'], errors='coerce')
            dm['gap']         = pd.to_numeric(dm['gap'],         errors='coerce')
            dm['label']       = dm.get('cell_line', dm.get('assigned_line', dm['vireo_donor']))
            conf_color = {'high': '#2d6a4f', 'low': '#f4a261',
                          'no_match': '#6d6875', 'no_data': '#adb5bd'}
            bar_colors = [conf_color.get(str(c), '#adb5bd') for c in dm.get('confidence', ['no_data'] * len(dm))]

            fig4 = make_subplots(
                rows=1, cols=2,
                subplot_titles=('Best concordance per donor', 'Gap (best − second best)'),
                horizontal_spacing=0.12
            )
            fig4.add_trace(go.Bar(
                x=dm['vireo_donor'], y=dm['concordance'],
                marker_color=bar_colors,
                text=dm['label'], textposition='outside',
                hovertemplate='%{x} → %{text}<br>Concordance: %{y:.3f}<extra></extra>'
            ), row=1, col=1)
            fig4.add_hline(y=_MIN_CONCORDANCE, line_dash='dash', line_color='grey',
                           annotation_text='min_concordance', row=1, col=1)

            fig4.add_trace(go.Bar(
                x=dm['vireo_donor'], y=dm['gap'],
                marker_color=bar_colors,
                hovertemplate='%{x}<br>Gap: %{y:.3f}<extra></extra>'
            ), row=1, col=2)
            fig4.add_hline(y=_MIN_GAP, line_dash='dash', line_color='grey',
                           annotation_text='min_gap', row=1, col=2)

            fig4.update_layout(
                title=f'Vireo → DB donor matching — {run_id}',
                height=420, showlegend=False
            )
            display(fig4)
        else:
            print('donor_matches.tsv not found or not applicable (guided mode runs skip this step).')

    _scrna_busy = [False]

    def render_scrna_cell(run_id):
        if _scrna_busy[0]:
            return
        _scrna_busy[0] = True
        try:
            clear_output(wait=False)
            display(demux_selector)
            render_scrna(run_id)
        finally:
            _scrna_busy[0] = False

    demux_selector.observe(lambda ch: render_scrna_cell(ch['new']), names='value')
    render_scrna_cell(demux_runs[-1])

## Concordance profiles
Per-donor concordance against all DB references (Plots A, B, C).
Only available for unguided runs (`pool_ctr_v2`, `pool_ctr_v4`); guided runs skip this step.

In [ ]:
sys.path.insert(0, os.path.abspath('scripts'))
from match_vireo import load_donor_genotypes, load_db_genotypes, compute_concordance as _compute_concordance

MIN_CONCORDANCE = 0.80
MIN_POS = 200

_ref_labels_cache = {}
def get_ref_labels(run_ids):
    if not _ref_labels_cache:
        rows = query("""
            SELECT r.run_id, COALESCE(s.cell_line, s.name) AS label
            FROM runs r JOIN samples s ON r.sample_id = s.sample_id
        """)
        _ref_labels_cache.update(dict(zip(rows['run_id'], rows['label'])))
    return [_ref_labels_cache.get(r, r) for r in run_ids]

_conc_cache = {}

def compute_conc(run_id):
    if run_id in _conc_cache:
        return _conc_cache[run_id]
    vireo_dir    = f'{DEMUX_DIR}/{run_id}/vireo'
    matches_path = f'{DEMUX_DIR}/{run_id}/donor_matches.tsv'
    if not os.path.exists(matches_path):
        print(f'{run_id}: donor_matches.tsv not found — concordance profiles are only available for unguided runs.')
        return None
    if not os.path.exists(vireo_dir):
        print(f'{run_id}: vireo directory not found.')
        return None
    print('Computing concordance matrix… (first load per run, ~20 s)')
    donors, positions, vireo_dosage = load_donor_genotypes(vireo_dir)
    ref_ids, db_dosage = load_db_genotypes(DB_PATH, ['all'], positions)
    concordance, n_shared = _compute_concordance(vireo_dosage, db_dosage)
    dm = pd.read_csv(matches_path, sep='\t')
    ref_labels = get_ref_labels(ref_ids)
    _conc_cache[run_id] = (donors, ref_ids, ref_labels, concordance, n_shared, dm)
    return _conc_cache[run_id]

if not demux_runs:
    print('No demux runs available.')
else:
    conc_selector = widgets.Dropdown(
        options=demux_runs,
        description='Demux run:',
        layout=widgets.Layout(width='350px')
    )
    _conc_busy = [False]

    def render_concordance(run_id):
        if _conc_busy[0]:
            return
        _conc_busy[0] = True
        try:
            clear_output(wait=False)
            display(conc_selector)

            result = compute_conc(run_id)
            if result is None:
                return
            donors, ref_ids, ref_labels, concordance, n_shared, dm = result

            n_donors = len(donors)

            best_c = np.array([
                np.nanmax(concordance[di]) if not np.all(np.isnan(concordance[di])) else 0.0
                for di in range(n_donors)
            ])

            dm_indexed = dm.set_index('vireo_donor')
            def donor_match_info(donor):
                if donor not in dm_indexed.index:
                    return None, None
                row = dm_indexed.loc[donor]
                return row.get('assigned_line', None), row.get('confidence', None)

            # ── Plot A: best vs worst donor ──────────────────────────────────
            best_di  = int(np.argmax(best_c))
            worst_di = int(np.argmin(best_c))

            def make_donor_bars(di):
                c_row = concordance[di].copy()
                invalid = np.isnan(c_row) | (n_shared[di] < MIN_POS)
                display_c = np.where(invalid, 0.0, c_row)
                order = np.argsort(display_c)[::-1]
                sorted_c      = display_c[order]
                sorted_labels = [ref_labels[i] for i in order]
                sorted_ids    = [ref_ids[i]    for i in order]
                assigned, conf = donor_match_info(donors[di])
                colors = []
                for orig_idx, rid in zip(order, sorted_ids):
                    if rid == assigned and conf == 'high':
                        colors.append('#2d6a4f')
                    elif rid == assigned and conf == 'low':
                        colors.append('#f4a261')
                    elif invalid[orig_idx]:
                        colors.append('#e9ecef')
                    else:
                        colors.append('#adb5bd')
                return sorted_c, sorted_labels, colors, assigned, conf

            fig_a = make_subplots(
                rows=1, cols=2,
                subplot_titles=(
                    f'{donors[best_di]} — best match',
                    f'{donors[worst_di]} — worst / no match'
                ),
                horizontal_spacing=0.12
            )
            fig_a_labels = []
            for col_idx, di in enumerate([best_di, worst_di], start=1):
                c_vals, labels, colors, assigned, conf = make_donor_bars(di)
                fig_a_labels.append(labels)
                fig_a.add_trace(go.Bar(
                    x=labels, y=c_vals, marker_color=colors,
                    hovertemplate='%{x}<br>Concordance: %{y:.3f}<extra></extra>',
                    showlegend=False
                ), row=1, col=col_idx)
                fig_a.add_hline(y=MIN_CONCORDANCE, line_dash='dash', line_color='#e63946',
                                annotation_text='threshold', row=1, col=col_idx)

            fig_a.update_yaxes(range=[0, 1], title_text='Concordance')
            fig_a.update_layout(height=480, title=f'Plot A: match vs no-match — {run_id}')
            for col_idx, lbs in enumerate(fig_a_labels, start=1):
                step = max(1, len(lbs) // 8)
                tickvals = [lbs[i] for i in range(0, len(lbs), step)]
                axis_key = 'xaxis' if col_idx == 1 else 'xaxis2'
                fig_a.update_layout(**{axis_key: dict(
                    tickmode='array', tickvals=tickvals, ticktext=tickvals, tickangle=45
                )})
            display(fig_a)

            # ── Plot B: 3-donor concordance profiles ─────────────────────────
            sorted_by_c = np.argsort(best_c)
            trio = [int(sorted_by_c[-1]), int(sorted_by_c[len(sorted_by_c) // 2]), int(sorted_by_c[0])]
            trio_labels = ['Best match', 'Median match', 'Worst / no match']

            fig_b = make_subplots(
                rows=3, cols=1,
                subplot_titles=[
                    f'{tl} — {donors[di]} (best concordance {best_c[di]:.3f})'
                    for tl, di in zip(trio_labels, trio)
                ],
                vertical_spacing=0.06,
                shared_xaxes=True
            )
            for row_idx, di in enumerate(trio, start=1):
                c_vals, lbs, colors, _, _ = make_donor_bars(di)
                fig_b.add_trace(go.Bar(
                    x=c_vals[::-1], y=lbs[::-1], orientation='h',
                    marker_color=colors[::-1],
                    hovertemplate='%{y}<br>Concordance: %{x:.3f}<extra></extra>',
                    showlegend=False
                ), row=row_idx, col=1)
                fig_b.add_vline(x=MIN_CONCORDANCE, line_dash='dash', line_color='#e63946',
                                row=row_idx, col=1)

            fig_b.update_xaxes(range=[0, 1], title_text='Concordance', row=3, col=1)
            fig_b.update_yaxes(tickmode='linear', dtick=1)
            fig_b.update_layout(
                height=max(600, 20 * len(ref_ids) * 3),
                title=f'Plot B: concordance profiles — {run_id}  (green=high, orange=low, grey=unmatched)'
            )
            display(fig_b)

            # ── Plot C: concordance score distributions ───────────────────────
            BINS = np.linspace(0, 1, 21)
            fig_c = make_subplots(
                rows=1, cols=3,
                subplot_titles=[f'{donors[di]}  ({trio_labels[idx]})' for idx, di in enumerate(trio)],
                horizontal_spacing=0.08
            )
            for col_idx, (idx, di) in enumerate(zip(range(3), trio), start=1):
                c_row   = concordance[di]
                valid_m = ~np.isnan(c_row) & (n_shared[di] >= MIN_POS)
                c_valid = c_row[valid_m]
                counts, edges = np.histogram(c_valid, bins=BINS)
                ymax = max(1, int(counts.max()))

                fig_c.add_trace(go.Bar(
                    x=(edges[:-1] + edges[1:]) / 2, y=counts,
                    width=(edges[1] - edges[0]) * 0.85,
                    marker_color='#adb5bd', showlegend=False,
                    customdata=np.column_stack([edges[:-1], edges[1:]]),
                    hovertemplate='Concordance %{customdata[0]:.2f}–%{customdata[1]:.2f}: %{y} refs<extra></extra>'
                ), row=1, col=col_idx)
                fig_c.add_trace(go.Scatter(
                    x=[MIN_CONCORDANCE, MIN_CONCORDANCE], y=[0, ymax * 1.2],
                    mode='lines', line=dict(color='#e63946', dash='dash', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ), row=1, col=col_idx)

                assigned_c, conf_c = donor_match_info(donors[di])
                if assigned_c and assigned_c != 'no_match' and assigned_c in ref_ids:
                    ari     = ref_ids.index(assigned_c)
                    match_c = float(c_row[ari]) if not np.isnan(c_row[ari]) else None
                    if match_c is not None:
                        line_color = '#2d6a4f' if conf_c == 'high' else '#f4a261'
                        fig_c.add_trace(go.Scatter(
                            x=[match_c, match_c], y=[0, ymax * 1.2],
                            mode='lines', line=dict(color=line_color, width=2),
                            showlegend=False, hoverinfo='skip'
                        ), row=1, col=col_idx)

            fig_c.update_xaxes(range=[0, 1], title_text='Concordance score')
            fig_c.update_yaxes(title_text='# DB references', row=1, col=1)
            fig_c.update_layout(
                height=320,
                title=(
                    f'Plot C: concordance distribution — {run_id}  |  '
                    'red dashed=threshold  |  green/orange=assigned match'
                )
            )
            display(fig_c)
        finally:
            _conc_busy[0] = False

    conc_selector.observe(lambda ch: render_concordance(ch['new']), names='value')
    render_concordance(demux_runs[-1])

## Validation — reference-guided Vireo (Pool_ctr)
Pool_ctr was run with known donor identities provided to Vireo.
Cell line assignments, Vireo confidence scores, and informative variant counts.

In [ ]:
_CELL_LINE_DISPLAY = {
    'rko': 'RKO', 'hct116': 'HCT116', 'ht29': 'HT29',
    'lim1215': 'LIM1215', 'caco2': 'CACO2', 'hct8': 'HCT8', 'dld1': 'DLD1',
    'doublet': 'doublet', 'unassigned': 'unassigned',
}

_val_path = 'validation/donor_ids.tsv'
if not os.path.exists(_val_path):
    print(f'Validation file not found: {_val_path}')
else:
    val = pd.read_csv(_val_path, sep='\t')
    val['donor_label'] = val['donor_id'].map(
        lambda x: _CELL_LINE_DISPLAY.get(str(x).lower(), str(x))
    )
    val['prob_max'] = pd.to_numeric(val['prob_max'], errors='coerce')
    val['n_vars']   = pd.to_numeric(val['n_vars'],   errors='coerce')

    _meta = {'doublet', 'unassigned'}
    _singlet_order = (
        val[~val['donor_label'].isin(_meta)]
        .groupby('donor_label').size()
        .sort_values(ascending=False).index.tolist()
    )
    _donor_order = _singlet_order + [d for d in ['doublet', 'unassigned'] if d in val['donor_label'].values]

    # ── 1. Cell composition bar ───────────────────────────────────────────────
    _comp = val['donor_label'].value_counts().reindex(_donor_order, fill_value=0)
    _bar_colors = [
        '#adb5bd' if d == 'unassigned' else
        '#e63946' if d == 'doublet'    else
        '#2d6a4f'
        for d in _donor_order
    ]
    fig_v1 = go.Figure(go.Bar(
        x=_comp.index, y=_comp.values,
        marker_color=_bar_colors,
        text=_comp.values, textposition='outside'
    ))
    fig_v1.update_layout(
        title=f'Cell composition — Pool_ctr reference-guided Vireo  ({len(val):,} cells)',
        xaxis_title='Donor / status', yaxis_title='Cell count',
        height=420, showlegend=False
    )
    fig_v1.show()

    # ── 2. prob_max violin per donor ──────────────────────────────────────────
    fig_v2 = go.Figure()
    for donor in _singlet_order:
        sub = val[val['donor_label'] == donor]['prob_max'].dropna()
        if len(sub) == 0:
            continue
        fig_v2.add_trace(go.Violin(y=sub, name=donor,
                                    box_visible=True, meanline_visible=True, points=False))
    fig_v2.update_layout(
        title='Vireo assignment probability (prob_max) per cell line — Pool_ctr',
        yaxis_title='prob_max', height=420, showlegend=False
    )
    fig_v2.show()

    # ── 3. n_vars distribution per donor ─────────────────────────────────────
    if 'n_vars' in val.columns:
        fig_v3 = go.Figure()
        for donor in _singlet_order:
            sub = val[val['donor_label'] == donor]['n_vars'].dropna()
            if len(sub) == 0:
                continue
            fig_v3.add_trace(go.Violin(y=sub, name=donor,
                                        box_visible=True, meanline_visible=True, points=False))
        fig_v3.update_layout(
            title='Number of informative variants per cell — Pool_ctr',
            yaxis_title='n_vars (SNPs used for assignment)', height=420, showlegend=False
        )
        fig_v3.show()

    # Summary table
    agg_dict = {'n_cells': ('cell', 'count'), 'mean_prob_max': ('prob_max', 'mean'),
                'median_prob_max': ('prob_max', 'median')}
    if 'n_vars' in val.columns:
        agg_dict['mean_n_vars'] = ('n_vars', 'mean')
    _summary_v = (
        val.groupby('donor_label').agg(**agg_dict)
        .reindex(_donor_order).reset_index()
    )
    fmt = {'mean_prob_max': '{:.3f}', 'median_prob_max': '{:.3f}'}
    if 'mean_n_vars' in _summary_v.columns:
        fmt['mean_n_vars'] = '{:.0f}'
    display(_summary_v.style.format(fmt))